# MAP5935 - Statistical Learning (Chapter 10 - Deep Learning)

**Prof. Christian Jäkel**

https://www.statlearning.com/

In [44]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots

from ISLP import load_data
from ISLP.models import ModelSpec as MS

from sklearn.linear_model import \
    (LinearRegression ,
    LogisticRegression ,
    Lasso)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline

from sklearn.model_selection import \
    (train_test_split ,
    GridSearchCV)

import torch
from torch import nn
from torch.optim import RMSprop
from torch.utils.data import TensorDataset

## *Conceptual Exercises*

### (3) Show that the negative multinomial log-likelihood $(1)$ is equivalent to the negative log of the likelihood expression $(2)$ when there are $M = 2$ classes.

### $$ -\sum_{i=1}^n\sum_{m=0}^1 y_{im}\log(f_m(x_i))\tag{1}$$

### $$ l(\beta_0, \beta_1)=\prod_{i:y_i=1}p(x_i)\prod_{i':y_{i'}=0}(1-p(x_{i'}))\tag{2}$$

**Binary case: cross-entropy = negative log-likelihood**

Let $M=2$ classes, coded as $y_i\in\{0,1\}$ with
$$
y_{i1}=\mathbb 1\{y_i=1\}=y_i,\qquad
y_{i0}=\mathbb 1\{y_i=0\}=1-y_i .
$$
Model probabilities:
$$
f_1(x_i)=p(x_i)=\Pr(Y=1\mid x_i),\qquad
f_0(x_i)=1-p(x_i)=\Pr(Y=0\mid x_i).
$$

**Cross-entropy (multinomial negative log-likelihood) (1):**
$$
-\sum_{i=1}^n \sum_{m\in\{0,1\}} y_{im}\log f_m(x_i)
= -\sum_{i=1}^n \Big[ y_i\log p(x_i) + (1-y_i)\log \big(1-p(x_i)\big) \Big]. \tag{Cross-Entropy}
$$

**Bernoulli likelihood (2):**
$$
\ell(\beta_0,\beta_1)
=\prod_{i:y_i=1} p(x_i)\;\prod_{i':y_{i'}=0}\big(1-p(x_{i'})\big).
$$

Taking logs and writing as a single sum,
$$
\log \ell(\beta_0,\beta_1)
=\sum_{i=1}^n \Big[ y_i\log p(x_i) + (1-y_i)\log \big(1-p(x_i)\big) \Big].
$$
Hence the **negative** log-likelihood is
$$
-\log \ell(\beta_0,\beta_1)
= -\sum_{i=1}^n \Big[ y_i\log p(x_i) + (1-y_i)\log \big(1-p(x_i)\big) \Big]. \tag{Negative Log-Likelihood}
$$

Therefore, for $M=2$ the multinomial cross-entropy in (1) reduces exactly to the negative log of the Bernoulli likelihood in (2), i.e., the standard binary logistic regression loss.


## *Applied Exercises*

### (9) Fit a lag-$5$ autoregressive model to the NYSE data, as described in the text and Lab $10.9.6$. Refit the model with a $12$-level factor representing the month. Does this factor improve the performance of the model?

**Solution** - According with the LAB $10.9.6$, we need do adapt it for a lag-$12$ representing the month for an RNN.

The following code was provided in the LAB.


In [24]:
NYSE = load_data('NYSE')
NYSE.columns

Index(['day_of_week', 'DJ_return', 'log_volume', 'log_volatility', 'train'], dtype='object')

In [25]:
NYSE.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6051 entries, 1962-12-03 to 1986-12-31
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   day_of_week     6051 non-null   category
 1   DJ_return       6051 non-null   float64 
 2   log_volume      6051 non-null   float64 
 3   log_volatility  6051 non-null   float64 
 4   train           6051 non-null   bool    
dtypes: bool(1), category(1), float64(3)
memory usage: 201.1+ KB


In [26]:
cols = ['DJ_return', 'log_volume', 'log_volatility']
X = pd.DataFrame(StandardScaler(
    with_mean=True ,
    with_std=True).fit_transform(NYSE[cols]),
                 columns=NYSE[cols].columns ,
                 index=NYSE.index)

In [27]:
for lag in range(1, 6):
    for col in cols:
        newcol = np.zeros(X.shape [0]) * np.nan
        newcol[lag:] = X[col].values[:-lag]
        X.insert(len(X.columns), "{0}_{1}".format(col , lag), newcol)
X.insert(len(X.columns), 'train', NYSE['train'])
X = X.dropna()

In [28]:
Y, train = X['log_volume'], X['train']
X = X.drop(columns=['train'] + cols)
X.columns

Index(['DJ_return_1', 'log_volume_1', 'log_volatility_1', 'DJ_return_2',
       'log_volume_2', 'log_volatility_2', 'DJ_return_3', 'log_volume_3',
       'log_volatility_3', 'DJ_return_4', 'log_volume_4', 'log_volatility_4',
       'DJ_return_5', 'log_volume_5', 'log_volatility_5'],
      dtype='object')

In [30]:
M = LinearRegression()
M.fit(X[train], Y[train])
M.score(X[~train], Y[~train])

0.41289129385625223

In [32]:
X_day = pd.merge(X, pd.get_dummies(NYSE['day_of_week']), on='date')

In [33]:
M.fit(X_day[train], Y[train])
M.score(X_day[~train], Y[~train])

0.4595563133053274

In [38]:
# As in LAB - Expect 5 lagged versions of each feature as indicated by the input_shape argument to the layer nn.RNN() below
ordered_cols = []
for lag in range(5,0,-1):
    for col in cols:
        ordered_cols.append('{0}_{1}'.format(col , lag))
X = X.reindex(columns=ordered_cols)
X.columns

Index(['DJ_return_5', 'log_volume_5', 'log_volatility_5', 'DJ_return_4',
       'log_volume_4', 'log_volatility_4', 'DJ_return_3', 'log_volume_3',
       'log_volatility_3', 'DJ_return_2', 'log_volume_2', 'log_volatility_2',
       'DJ_return_1', 'log_volume_1', 'log_volatility_1'],
      dtype='object')

In [39]:
# Reshape the Data
X_rnn = X.to_numpy().reshape((-1,5,3))
X_rnn.shape

(6046, 5, 3)

In [41]:
X_rnn

array([[[-0.54982334,  0.17507497, -4.35707786],
        [ 0.90519995,  1.51729071, -2.52905765],
        [ 0.43481275,  2.28378937, -2.41803694],
        [-0.43139673,  0.93517558, -2.36652094],
        [ 0.04634026,  0.22477858, -2.5009701 ]],

       [[ 0.90519995,  1.51729071, -2.52905765],
        [ 0.43481275,  2.28378937, -2.41803694],
        [-0.43139673,  0.93517558, -2.36652094],
        [ 0.04634026,  0.22477858, -2.5009701 ],
        [-1.30412619,  0.60591805, -1.366028  ]],

       [[ 0.43481275,  2.28378937, -2.41803694],
        [-0.43139673,  0.93517558, -2.36652094],
        [ 0.04634026,  0.22477858, -2.5009701 ],
        [-1.30412619,  0.60591805, -1.366028  ],
        [-0.00629379, -0.01366095, -1.50566722]],

       ...,

       [[ 0.96826597,  4.25840159,  0.26340567],
        [-0.18517843,  1.6026693 ,  0.12800392],
        [-0.75004612,  1.96484574,  0.08025041],
        [ 0.75120978, -0.97476289,  0.04688631],
        [ 0.19535153, -5.62381367, -0.08398311]],


Now we are ready to proceed with the *RNN*, which uses $12$ hidden units, and 10% dropout. After passing through the *RNN*, we extract the final time point as `val[:,-1]` in `forward()` below. This gets passed through a 10% dropout and then flattened through a linear layer

In [45]:
class NYSEModel(nn.Module):
    def __init__(self):
        super(NYSEModel , self).__init__()
        self.rnn = nn.RNN(3,
                          12,
                          batch_first=True)
        self.dense = nn.Linear(12, 1)
        self.dropout = nn.Dropout (0.1)
    def forward(self , x):
        val , h_n = self.rnn(x)
        val = self.dense(self.dropout(val[:,-1]))
        return torch.flatten(val)
nyse_model = NYSEModel()